# 🎙️ IELTS 스피킹 영상 메이커
### OpenVoice 내 목소리 클로닝 버전

**실행 전 체크:**
- 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
- 셀을 위에서부터 순서대로 실행하세요 (Shift+Enter)


## 셀 1 — 패키지 설치 (최초 1회만, 5~10분 소요)

In [ ]:
# FIX 1: 빌드 도구를 가장 먼저 설치 (fugashi, numpy 등 C 확장 빌드에 필요)
!apt-get install -y build-essential python3-dev libmecab-dev mecab mecab-ipadic-utf8 -q

# FIX 2: pip/setuptools/wheel 최신화 및 Cython 선행 설치
!pip install --upgrade pip setuptools wheel Cython -q

# FIX 3: C 확장 패키지를 단독으로 먼저 빌드 (MeloTTS 설치 전에 환경 확보)
!pip install numpy fugashi -q

# FIX 4: %cd 제거 → 절대경로로 설치 (working directory 오염 방지)
import os
if not os.path.exists('/content/OpenVoice'):
    !git clone https://github.com/myshell-ai/OpenVoice.git /content/OpenVoice
else:
    print('OpenVoice 이미 존재, 클론 스킵')
!pip install -e /content/OpenVoice -q

# FIX 5: MeloTTS 설치 후 unidic 다운로드 (순서 중요)
!pip install git+https://github.com/myshell-ai/MeloTTS.git -q
!python -m unidic download

# FIX 6: gTTS ↔ click 버전 충돌 해결 (click<8.2 명시)
!pip install "gTTS>=2.5.0" "click>=8.0,<8.2" -q

# 기타 필요 패키지
!pip install faster-whisper streamlit pyngrok google-generativeai moviepy Pillow -q

print('✅ 설치 완료')

## 셀 2 — OpenVoice 모델 다운로드 (최초 1회만, 2~3분 소요)

In [ ]:
import os, zipfile, urllib.request

checkpoint_url = 'https://myshell-public-repo-host.s3.amazonaws.com/openvoice/checkpoints_v2_0417.zip'
zip_path = '/content/checkpoints_v2.zip'
ckpt_dir = '/content/OpenVoice/checkpoints_v2'

if not os.path.exists(ckpt_dir):
    print('📥 모델 다운로드 중...')
    urllib.request.urlretrieve(checkpoint_url, zip_path)
    print('📦 압축 해제 중...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/OpenVoice/')
    os.remove(zip_path)
    print('✅ 모델 준비 완료')
else:
    print('✅ 이미 다운로드됨')

# FIX 7: 체크포인트 파일 존재 여부 검증
required_files = [
    f'{ckpt_dir}/converter/config.json',
    f'{ckpt_dir}/converter/checkpoint.pth',
    f'{ckpt_dir}/base_speakers/ses/en-us.pth',
]
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    print('❌ 누락된 파일:'); [print(f'   {f}') for f in missing]
    print('   → 셀을 다시 실행하거나 zip URL을 확인하세요.')
else:
    print('✅ 모든 체크포인트 파일 확인 완료')

## 셀 3 — Gemini API Key 입력

In [ ]:
# @title Gemini API Key 입력
# aistudio.google.com 에서 무료 발급 (카드 불필요)
GEMINI_API_KEY = ''  # @param {type:"string"}

# FIX 8: API Key 형식 기초 검증
if not GEMINI_API_KEY:
    print('⚠️  API Key를 입력해주세요')
elif not GEMINI_API_KEY.startswith('AI'):
    print('⚠️  API Key 형식이 올바르지 않습니다. aistudio.google.com에서 다시 확인해주세요.')
else:
    print('✅ API Key 설정 완료')

## 셀 4 — 내 목소리 파일 업로드

In [ ]:
import os
from google.colab import files
import shutil

print('📂 내 목소리 녹음 파일을 업로드하세요 (MP3 또는 WAV, 10초 이상)')
print('   조용한 환경 · 또박또박 발음 · 배경 소음 최소화')
print()

uploaded = files.upload()

if uploaded:
    ref_filename = list(uploaded.keys())[0]
    ext = os.path.splitext(ref_filename)[1].lower()

    # FIX 9: 지원 형식 검증
    if ext not in ['.mp3', '.wav', '.m4a', '.flac', '.ogg']:
        print(f'⚠️  지원하지 않는 형식입니다: {ext}')
        print('   MP3, WAV, M4A, FLAC, OGG 파일을 사용해주세요.')
        ref_path = None
    else:
        ref_path = f'/content/my_voice{ext}'
        shutil.copy(ref_filename, ref_path)
        size_kb = os.path.getsize(ref_path) / 1024
        print(f'✅ 업로드 완료: {ref_filename} ({size_kb:.0f} KB)')
        print(f'   저장 위치: {ref_path}')
        # FIX 10: 파일 너무 작으면 경고 (10초 미만 가능성)
        if size_kb < 50:
            print('⚠️  파일이 너무 작습니다. 10초 이상 녹음하셨나요?')
else:
    ref_path = None
    print('❌ 파일이 업로드되지 않았습니다')

## 셀 5 — 핵심 함수 정의 (한 번만 실행)

In [ ]:
import os, json, sys, tempfile
import numpy as np
import torch
import google.generativeai as genai
from gtts import gTTS
from PIL import Image, ImageDraw, ImageFont

# FIX 11: ref_path 정의 여부 사전 확인 (셀 4 건너뛰기 방지)
try:
    ref_path
except NameError:
    raise RuntimeError('❌ ref_path가 없습니다. 셀 4(목소리 업로드)를 먼저 실행해주세요.')
if ref_path is None or not os.path.exists(ref_path):
    raise RuntimeError('❌ 목소리 파일이 없습니다. 셀 4를 다시 실행해주세요.')

# FIX 12: GEMINI_API_KEY 정의 여부 사전 확인 (셀 3 건너뛰기 방지)
try:
    GEMINI_API_KEY
except NameError:
    raise RuntimeError('❌ GEMINI_API_KEY가 없습니다. 셀 3을 먼저 실행해주세요.')
if not GEMINI_API_KEY:
    raise RuntimeError('❌ GEMINI_API_KEY가 비어 있습니다. 셀 3에서 키를 입력해주세요.')

sys.path.insert(0, '/content/OpenVoice')
from openvoice import se_extractor
from openvoice.api import ToneColorConverter
from melo.api import TTS as MeloTTS

# ── Gemini 설정 ───────────────────────────────────────────────────────────────
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel('gemini-2.0-flash')

score_desc = {
    5.0: '기초 – 단순 문장, 기본 어휘',
    5.5: '초중급 – 조금 더 자연스럽게',
    6.0: '중급 – 복잡한 구조 시작',
    6.5: '중상급 – 다양한 어휘, 자연스러운 흐름',
    7.0: '상급 – 관용구, 고급 어휘 포함',
    7.5: '고급 – 원어민에 가까운 표현',
    8.0: '최고급 – 네이티브 수준',
}

# ── OpenVoice 모델 로드 ───────────────────────────────────────────────────────
print('🔄 OpenVoice 모델 로드 중... (첫 실행 시 1~2분 소요)')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'   사용 디바이스: {device}')
if device == 'cpu':
    print('⚠️  GPU가 감지되지 않습니다. 런타임 유형을 T4 GPU로 변경하면 훨씬 빠릅니다.')

ckpt_converter = '/content/OpenVoice/checkpoints_v2/converter'
tone_converter = ToneColorConverter(f'{ckpt_converter}/config.json', device=device)
tone_converter.load_ckpt(f'{ckpt_converter}/checkpoint.pth')
print('✅ OpenVoice 모델 로드 완료')

# ── 내 목소리 특성 추출 ───────────────────────────────────────────────────────
print('🎙️ 내 목소리 특성 분석 중...')
target_se, _ = se_extractor.get_se(ref_path, tone_converter, vad=True)
print('✅ 목소리 특성 추출 완료')

# ── MeloTTS 로드 (베이스 음성 생성용) ────────────────────────────────────────
print('🔄 MeloTTS 로드 중...')
melo_model = MeloTTS(language='EN', device=device)
speaker_ids = melo_model.hps.data.spk2id
print('✅ MeloTTS 로드 완료')

# FIX 13: EN-US 스피커 존재 여부 확인
if 'EN-US' not in speaker_ids:
    available = list(speaker_ids.keys())
    print(f'⚠️  EN-US 스피커 없음. 사용 가능: {available}')
    EN_SPEAKER = available[0]
    print(f'   → {EN_SPEAKER} 으로 대체합니다.')
else:
    EN_SPEAKER = 'EN-US'

print()
print('🎉 모든 모델 준비 완료! 셀 6으로 이동하세요.')

# ── Gemini 스크립트 생성 함수 ─────────────────────────────────────────────────
def generate_script(question, korean_answer, target_score):
    # FIX 14: target_score가 score_desc에 없으면 가장 가까운 값으로 대체
    valid_scores = sorted(score_desc.keys())
    if target_score not in score_desc:
        target_score = min(valid_scores, key=lambda x: abs(x - target_score))
        print(f'⚠️  유효하지 않은 점수 → {target_score}로 대체')

    prompt = f"""You are an IELTS speaking coach. Transform the Korean answer into natural English for IELTS Band {target_score}.

Band {target_score} guidance: {score_desc[target_score]}
Rules:
- Vocabulary and grammar complexity appropriate for Band {target_score}
- Natural and conversational, not robotic
- Length: 60-120 words for Part 1, 150-200 for Part 2, 80-120 for Part 3
- Use discourse markers (Well, Actually, To be honest, etc.)

IELTS Question: {question}
Korean answer: {korean_answer}

Respond ONLY with raw JSON (no markdown, no code block):
{{"english": "...", "korean": "자연스러운 한국어 번역", "band_tips": "특징적 표현 설명 (한국어)"}}"""

    # FIX 15: Gemini 응답 파싱 강화 + 예외 처리
    try:
        raw = gemini_model.generate_content(prompt).text.strip()
        # 마크다운 코드블록 제거 (```json ... ``` 또는 ``` ... ```)
        if '```' in raw:
            parts = raw.split('```')
            # 짝수 인덱스가 블록 바깥, 홀수가 블록 안
            raw = parts[1] if len(parts) > 1 else raw
            if raw.startswith('json'):
                raw = raw[4:]
        # JSON 객체 범위만 추출
        start = raw.find('{')
        end   = raw.rfind('}')
        if start == -1 or end == -1:
            raise ValueError('JSON 객체를 찾을 수 없음')
        return json.loads(raw[start:end+1])
    except json.JSONDecodeError as e:
        raise RuntimeError(f'❌ Gemini 응답 파싱 실패: {e}\n원본:\n{raw}')
    except Exception as e:
        raise RuntimeError(f'❌ Gemini API 오류: {e}')

# ── OpenVoice 음성 생성 함수 ──────────────────────────────────────────────────
def generate_voice(text, output_path, speed=1.0):
    tmp_base = output_path.replace('.wav', '_base.wav')
    try:
        # MeloTTS로 베이스 음성 생성
        melo_model.tts_to_file(text, speaker_ids[EN_SPEAKER], tmp_base, speed=speed)

        # FIX 16: 베이스 음성 생성 확인 후 변환 진행
        if not os.path.exists(tmp_base):
            raise RuntimeError('베이스 음성 파일이 생성되지 않았습니다.')

        # OpenVoice로 내 목소리로 변환
        src_se_path = '/content/OpenVoice/checkpoints_v2/base_speakers/ses/en-us.pth'
        src_se = torch.load(src_se_path, map_location=device)
        tone_converter.convert(
            audio_src_path=tmp_base,
            src_se=src_se,
            tgt_se=target_se,
            output_path=output_path,
            message='@MyShell'
        )
    finally:
        # FIX 17: 오류 발생 여부와 무관하게 임시 파일 정리
        if os.path.exists(tmp_base):
            os.remove(tmp_base)

# ── 배경 프레임 생성 함수 ─────────────────────────────────────────────────────
def make_frame(style, label, text, target_score):
    W, H = 1280, 720
    img  = Image.new('RGB', (W, H))
    draw = ImageDraw.Draw(img)
    palettes = {
        '다크 미니멀': ((15, 15, 25),   (196, 149, 106), (235, 235, 235), (150, 150, 160)),
        '라이트 클린': ((248, 246, 241), (70, 90, 160),   (30, 30, 40),   (120, 120, 130)),
        '딥 블루':     ((10, 30, 60),    (100, 180, 255), (230, 240, 255), (140, 170, 210)),
    }
    bg, accent, fg, muted = palettes.get(style, palettes['다크 미니멀'])
    draw.rectangle([0, 0, W, H], fill=bg)
    draw.rectangle([60, 80, 80, H-80], fill=accent)
    try:
        fB = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 18)
        fM = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 28)
        fS = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 14)
    except:
        fB = fM = fS = ImageFont.load_default()
    draw.text((110, 90), label.upper(), fill=accent, font=fB)

    # FIX 18: 텍스트 줄바꿈 로직 — 픽셀 폭 기준으로 교체 (문자 수 52 기준은 폰트마다 달라 부정확)
    words = text.split()
    lines, cur = [], []
    for w in words:
        test_line = ' '.join(cur + [w])
        bbox = draw.textbbox((0, 0), test_line, font=fM)
        if bbox[2] > W - 180 and cur:   # 오른쪽 여백 180px 확보
            lines.append(' '.join(cur))
            cur = [w]
        else:
            cur.append(w)
    if cur:
        lines.append(' '.join(cur))

    y = 140
    for line in lines[:6]:
        draw.text((110, y), line, fill=fg, font=fM)
        y += 44
    # FIX 19: 텍스트가 잘렸을 때 '...' 표시
    if len(lines) > 6:
        draw.text((110, y), '...', fill=muted, font=fM)

    draw.text((W-230, H-50), f'IELTS Band {target_score}', fill=muted, font=fS)
    return np.array(img)

print('✅ 함수 정의 완료')

## 셀 6 — 영상 생성 (매번 실행)
아래 변수만 바꿔서 실행하면 새 영상이 만들어집니다.

In [ ]:
# ════════════════════════════════════════════
#  ✏️  여기만 수정하세요
# ════════════════════════════════════════════

QUESTION = "Tell me about your hometown. What do you like most about it?"

KOREAN_ANSWER = """
저는 부산 출신인데요, 바다가 정말 가까워서 좋아요.
어릴 때 아버지랑 주말마다 해운대에 갔던 기억이 있어요.
음식도 맛있고, 특히 해산물 요리가 유명해요.
"""

TARGET_SCORE = 6.5   # 5.0 / 5.5 / 6.0 / 6.5 / 7.0 / 7.5 / 8.0
VIDEO_STYLE  = '다크 미니멀'   # '다크 미니멀' / '라이트 클린' / '딥 블루'
OUTPUT_FILE  = '/content/ielts_speaking.mp4'

# ════════════════════════════════════════════
#  🚀 실행 (수정 불필요)
# ════════════════════════════════════════════

# FIX 20: moviepy 버전에 따라 import 경로가 다름 — 양쪽 시도
try:
    from moviepy.editor import AudioFileClip, ImageClip, concatenate_videoclips
except ImportError:
    from moviepy import AudioFileClip, ImageClip, concatenate_videoclips

import tempfile, os

# FIX 21: 셀 5 실행 여부 확인
for var_name, var in [('tone_converter', 'tone_converter'), ('melo_model', 'melo_model'),
                       ('target_se', 'target_se'), ('gemini_model', 'gemini_model')]:
    if var_name not in dir():
        raise RuntimeError(f'❌ {var_name}가 없습니다. 셀 5(핵심 함수 정의)를 먼저 실행해주세요.')

tmpdir = tempfile.mkdtemp()

# 1. 영어 스크립트 생성
print('✍️  영어 스크립트 생성 중...')
result = generate_script(QUESTION, KOREAN_ANSWER, TARGET_SCORE)
english_script = result['english']
print(f'\n📝 영어 스크립트:\n{english_script}')
print(f'\n🇰🇷 한국어 번역:\n{result["korean"]}')
print(f'\n💡 Band 포인트:\n{result["band_tips"]}')

# 2. 음성 생성 (내 목소리)
print('\n🎙️  내 목소리로 음성 생성 중...')
q_audio_path = os.path.join(tmpdir, 'question.wav')
a_audio_path = os.path.join(tmpdir, 'answer.wav')

print('   질문 음성 생성 중...')
generate_voice(QUESTION, q_audio_path)

print('   답변 음성 생성 중...')
generate_voice(english_script, a_audio_path)

# 3. 배경 프레임 생성
print('\n🖼️  배경 프레임 생성 중...')
q_frame = make_frame(VIDEO_STYLE, 'Question', QUESTION,       TARGET_SCORE)
a_frame = make_frame(VIDEO_STYLE, 'Answer',   english_script, TARGET_SCORE)

# 4. 영상 합성
print('\n🎞️  영상 합성 중...')
q_audio = AudioFileClip(q_audio_path)
a_audio = AudioFileClip(a_audio_path)
q_clip  = ImageClip(q_frame).set_duration(q_audio.duration + 1.5).set_audio(q_audio)
a_clip  = ImageClip(a_frame).set_duration(a_audio.duration + 2.0).set_audio(a_audio)
final   = concatenate_videoclips([q_clip, a_clip], method='compose')
final.write_videofile(OUTPUT_FILE, fps=24, codec='libx264', audio_codec='aac', logger=None)

# FIX 22: 리소스 해제를 try/finally로 보장
try:
    pass
finally:
    q_audio.close()
    a_audio.close()
    final.close()

# FIX 23: 출력 파일 존재 확인 후 다운로드
if os.path.exists(OUTPUT_FILE):
    size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
    print(f'\n🎉 영상 완성! → {OUTPUT_FILE} ({size_mb:.1f} MB)')
    from google.colab import files
    files.download(OUTPUT_FILE)
    print('⬇️  다운로드 시작됨')
else:
    print('❌ 영상 파일이 생성되지 않았습니다. 위 로그를 확인해주세요.')

---
## 참고: 질문 목록

**Part 1 – 일상**
- Tell me about your hometown. What do you like most about it?
- Do you enjoy cooking? Why or why not?
- How do you usually spend your weekends?
- What kind of music do you like? Why?

**Part 2 – 경험**
- Describe a memorable trip you have taken.
- Describe a time when you helped someone.
- Describe an achievement you are proud of.

**Part 3 – 사회/교육**
- Do you think technology has improved education? In what ways?
- How important is it for young people to learn a second language?
- What are the advantages and disadvantages of working from home?

---
셀 6의 `QUESTION`과 `KOREAN_ANSWER`만 바꿔서 계속 실행하면 됩니다.